In [11]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import  Chroma
from langchain_core.prompts import PromptTemplate
from apps.qna_bot import llm_model

In [12]:
#   Load ENV
load_dotenv()

True

In [13]:
#Doc Loading

pdf_loader = PyPDFLoader('../data/data_science_syllabus.pdf')
docs = pdf_loader.load()

In [14]:
#Doc Chunks
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
split_data = doc_splitter.split_documents(docs)

In [15]:
#Embeddings
try:
    embeddings  # Check if variable already exists
    print("Embeddings already loaded. Skipping...")
except NameError:
    print("Loading embeddings for the first time...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

Loading embeddings for the first time...


/Users/somit17/Documents/Project/GenAI/gen-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9485.17it/s]


In [16]:
#Vectore Store
vector_store = Chroma.from_documents(
    documents=split_data,
    embedding=embeddings,
    persist_directory="vector_db_store/chroma.db"
)

In [17]:
# query = "Machine Learning and Data Science Content"
#
# data = vector_store.similarity_search(query=query)
#print(data)

In [18]:
# context = ''
# for doc in data:
#     context+= doc.page_content + '\n'

In [19]:
llm_model = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    streaming=True
    # other params...
)

In [20]:
# res = llm_model.invoke(f"Can you provide me the answer based on provided context for my question,context : {context} and question = {query}")

In [21]:
# print(res.content)

In [22]:
### CHAIN  - Context_generate -> prompt -> llm -> output

In [23]:
def get_context(user_query: str):
    # k=5 retrieves the top 5 most relevant chunks
    data = vector_store.similarity_search(query=user_query, k=5)

    # Join with clear separators
    context = '\n\n---\n\n'.join([doc.page_content for doc in data])

    # DEBUG: Print exactly what is being sent to the LLM
    print("\n=== 🕵️ DEBUG: RETRIEVED CONTEXT ===")
    print(context if context else "NO CONTEXT FOUND!")
    print("====================================\n")

    return {"context": context, "question": user_query}


In [24]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Answer the user's question based ONLY on the provided context.
If the answer is not explicitly in the context, simply reply 'I dont know'.

<context>
{context}
</context>

<question>
{question}
</question>

Answer:
""")

In [25]:
from langchain_core.runnables import RunnableLambda

rag_chain = RunnableLambda(get_context) | prompt | llm_model

In [26]:
res = rag_chain.invoke("What are the mini projects in Module 1 ?Give me names")


=== 🕵️ DEBUG: RETRIEVED CONTEXT ===
✅  Module 4: Data Visualization & Storytelling (3 
weeks)
Duration: Month 4
Topics:
Matplotlib, Seaborn, Plotly
Creating interactive dashboards
Power BI or Tableau (choose one)
Storytelling with data
Tools:
Power BI
Mini Project:
Power BI Dashboard (e.g. COVID, Sales, HR analytics)
🎤  Mock Interview: Visualization Tools  Data Storytelling
✅  Module 5: Statistics & Probability for Data Science 
(4 weeks)
Duration: Month 5
Topics:
Descriptive statistics
Probability distributions
Sampling, CLT

---

✅  Module 4: Data Visualization & Storytelling (3 
weeks)
Duration: Month 4
Topics:
Matplotlib, Seaborn, Plotly
Creating interactive dashboards
Power BI or Tableau (choose one)
Storytelling with data
Tools:
Power BI
Mini Project:
Power BI Dashboard (e.g. COVID, Sales, HR analytics)
🎤  Mock Interview: Visualization Tools  Data Storytelling
✅  Module 5: Statistics & Probability for Data Science 
(4 weeks)
Duration: Month 5
Topics:
Descriptive statistics
Pro

In [27]:
print(res.content)

I dont know
